# Colombia Open — Hurto a personas

Este notebook muestra cómo cargar el dataset **`delincuencia_hurto_personas`** (víctimas de hurto a personas por municipio y día, 2003 a la fecha, MinDefensa / Policía Nacional) con `colombia-open`.

> **Granularidad:** la fuente ya viene agregada — una fila por fecha + municipio con el número de víctimas de ese día (~650,835 filas). No es microdato por hecho: no trae sexo, edad, arma ni hora. Los días sin víctimas en un municipio no aparecen como fila con cero.

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.5.0`.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
hurto = hub.load_polars("delincuencia_hurto_personas")

print(hurto.shape)
hurto.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | date | Fecha del hecho. |
| `anio` | int | Año de `fecha` — para joins con datasets anuales como `demografia`. |
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Códigos DIVIPOLA — join key hacia `municipios`. |
| `nombre_dpto` / `nombre_mpio` | string | Nombres tal como los entrega la fuente (mayúsculas, sin tildes). |
| `cantidad` | int | Víctimas de hurto a personas ese día en ese municipio (siempre > 0). |

Clave primaria: `(fecha, cod_mpio)`. Esquema completo: [`docs/datasets/delincuencia_hurto_personas.md`](../../docs/datasets/delincuencia_hurto_personas.md).

## Serie anual de un municipio

In [ ]:
# Víctimas por año en Medellín (cod_mpio == "05001")
(
    hurto.filter(pl.col("cod_mpio") == "05001")
    .group_by("anio")
    .agg(pl.col("cantidad").sum().alias("victimas"))
    .sort("anio")
)

## Serie mensual nacional

In [ ]:
mensual = (
    hurto.group_by(pl.col("fecha").dt.truncate("1mo").alias("mes"))
    .agg(pl.col("cantidad").sum().alias("victimas"))
    .sort("mes")
)
mensual.tail(12)

## Tasa por 100 mil habitantes (join con `demografia`)

`demografia` (proyecciones DANE) cubre 2018-2042, así que la tasa se puede calcular desde 2018. Se usa el área geográfica `"Total"` y se cruza por `(cod_mpio, anio)`.

In [ ]:
poblacion = hub.load_polars("demografia").filter(pl.col("area_geografica") == "Total")

tasas = (
    hurto.group_by(["cod_mpio", "anio"])
    .agg(pl.col("cantidad").sum().alias("victimas"))
    .join(poblacion.select(["cod_mpio", "nombre_mpio", "anio", "poblacion"]), on=["cod_mpio", "anio"])
    .with_columns((pl.col("victimas") / pl.col("poblacion") * 100_000).round(1).alias("tasa_100k"))
)

# Municipios de más de 500 mil habitantes con mayor tasa en 2025
tasas.filter((pl.col("anio") == 2025) & (pl.col("poblacion") > 500_000)).sort(
    "tasa_100k", descending=True
).head(10)

## Warnings de validación y el defecto conocido de 2008

En 2008 la fuente casi no desagrega por municipio: trae una fila diaria con `cod_muni = "SIN ESTABLECER"` (365 filas, 64,185 víctimas), que el pipeline **excluye** porque no tienen join posible con `municipios`. El validador detecta dinámicamente el año con cobertura municipal anómala y lo reporta como **warning** — visible en el catálogo:

In [ ]:
for warning in hub.get_dataset("delincuencia_hurto_personas")["warnings"]:
    print("warning:", warning)

hurto.group_by("anio").agg(pl.col("cod_mpio").n_unique().alias("municipios")).sort("anio")

## Limitaciones conocidas

- **No compares 2008 con otros años**: su total nacional subestima el real en ~64 mil víctimas (filas "SIN ESTABLECER" excluidas).
- **La serie no es homogénea en el tiempo**: los municipios con registros pasan de ~630 (2003) a ~1050 (2023), y parte del crecimiento refleja cambios en el registro y la denuncia, no solo en el delito.
- **El año en curso está incompleto**: la fuente se actualiza con corte mensual.
- Solo hurto a personas — las lesiones personales están en `delincuencia_lesiones_personales` (ver `06_delincuencia_lesiones_personales.ipynb`); hurto por modalidades no está incluido todavía.

Documentación completa: [`docs/datasets/delincuencia_hurto_personas.md`](../../docs/datasets/delincuencia_hurto_personas.md).